## Comparación de clasificadores

In [ ]:
# Preparación: librerías y utilidades
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, auc, ConfusionMatrixDisplay)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Configurar impresión y gráficos
np.set_printoptions(suppress=True)
pd.set_option("display.max_columns", 100)

In [ ]:
# Ejemplo: Clasificación con Iris
# - Usaremos dos características para visualizar en 2D: Petal length (cm) y Petal width (cm)
# - Convertimos a un problema binario para incluir ROC (setosa vs no-setosa) y luego multiclase para comparar modelos

iris = load_iris(as_frame=True)
df = iris.frame.copy()
df.head()

In [ ]:
# Seleccionamos dos características para la visualización 2D
X = df[["petal length (cm)", "petal width (cm)"]].values
y = iris.target.values  # multiclase: 0=setosa, 1=versicolor, 2=virginica

# División train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

# Nota sobre stratify
# stratify=y: Este argumento se utiliza para la división estratificada.
# Significa que la función dividirá los datos de manera que se preserve la
# proporción de cada clase en el conjunto de datos original (y).
# Esto es particularmente importante para los problemas de clasificación,
#especialmente cuando tienes clases desequilibradas, para asegurar que tanto
# los conjuntos de entrenamiento como los de prueba sean representativos de
# la distribución general de clases.

In [ ]:
# Comparación de clasificadores (multiclase, macro-F1 con validación cruzada)
models = {
    "LogisticRegression": LogisticRegression(max_iter=1000, random_state=42),
    "KNN (k=5)": KNeighborsClassifier(n_neighbors=5),
    "SVC (RBF)": SVC(kernel="rbf", probability=True, random_state=42),
    "DecisionTree": DecisionTreeClassifier(random_state=42),
    "RandomForest": RandomForestClassifier(n_estimators=200, random_state=42)
}



In [ ]:
# El escalado se ajustará dentro de cada pliegue mediante un pipeline.

## Validacion cruzada

La validación cruzada (cross-validation) es una técnica utilizada en machine learning para evaluar el rendimiento de un modelo predictivo de manera más robusta que una simple división de datos en conjuntos de entrenamiento y prueba.

Imagina que tienes un conjunto de datos y quieres entrenar un modelo. Si solo lo entrenas en una parte de los datos y lo pruebas en otra, el rendimiento que observes puede ser muy sensible a la forma en que dividiste los datos. Si tienes "suerte" con la división, el rendimiento en el conjunto de prueba podría ser engañosamente bueno (o malo).

La validación cruzada aborda esto dividiendo el conjunto de datos en varios "pliegues" o "folds" (generalmente k pliegues, de ahí el nombre "k-fold cross-validation"). El proceso es el siguiente:

- División en k pliegues: El conjunto de datos se divide en k subconjuntos de tamaño aproximadamente igual.
Iteración: El proceso se repite k veces. En cada iteración:
Uno de los pliegues se utiliza como conjunto de prueba (validación).

Los k-1 pliegues restantes se combinan para formar el conjunto de entrenamiento.
Entrenamiento y evaluación: El modelo se entrena con el conjunto de entrenamiento y se evalúa con el conjunto de prueba de esa iteración. Se registra la métrica de rendimiento (por ejemplo, accuracy, F1-score, etc.).

- Resultados: Al finalizar las k iteraciones, tienes k resultados de rendimiento, uno por cada pliegue utilizado como conjunto de prueba.
Finalmente, se suele calcular la media y la desviación estándar de estas k métricas de rendimiento. La media te da una estimación más confiable del rendimiento esperado del modelo en datos no vistos, ya que se ha probado en diferentes subconjuntos del conjunto de datos original. La desviación estándar te indica cuánto varía el rendimiento entre las diferentes divisiones, lo que puede ser útil para entender la estabilidad del modelo.

En resumen, la validación cruzada ayuda a:

Obtener una estimación más fiable del rendimiento del modelo.
Detectar si el modelo es demasiado sensible a la división específica de los datos (alta varianza).
Hacer un mejor uso de todos los datos disponibles tanto para entrenamiento como para validación.

Este código compara diferentes modelos de clasificación (Regresión Logística, K-Vecinos Cercanos, SVC, Árbol de Decisión y Bosque Aleatorio) utilizando validación cruzada para evaluar su rendimiento con la métrica F1-macro.

## Comparación con validación cruzada
Cada pipeline aprende su escalador solamente con el entrenamiento de cada pliegue. Se selecciona el mayor F1 macro medio. El test se mantiene reservado hasta evaluar el modelo seleccionado.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold
pipelines = {name: make_pipeline(StandardScaler(), clone(clf)) for name, clf in models.items()}
# El número de pliegues no puede superar la frecuencia de la clase menos frecuente.
n_splits = min(5, int(pd.Series(y_train).value_counts().min()))
cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
rows = []
for name, pipeline in pipelines.items():
    scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='f1_macro')
    rows.append({'Modelo': name, 'F1_macro (media CV)': scores.mean(), 'F1_macro (std)': scores.std()})
tabla = pd.DataFrame(rows).sort_values('F1_macro (media CV)', ascending=False).reset_index(drop=True)
print(f'Validación cruzada estratificada: {n_splits} pliegues')
display(tabla)


In [ ]:
# 4.2 Visualización de la frontera de decisión en 2D (Árbol)
def plot_decision_regions(model, X, y, title, scaler=None, h=0.02):
    # si se pasa scaler, la malla se genera en espacio original pero se transforma para predecir
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    grid = np.c_[xx.ravel(), yy.ravel()]
    grid_scaled = scaler.transform(grid) if scaler is not None else grid
    Z = model.predict(grid_scaled if scaler is not None else grid)
    Z = Z.reshape(xx.shape)

    fig, ax = plt.subplots(figsize=(6,5))
    ax.contourf(xx, yy, Z, alpha=0.3)  # no especificar colores
    # puntos de entrenamiento
    for cls in np.unique(y):
        ax.scatter(X[y==cls,0], X[y==cls,1], label=f"Clase {cls}", edgecolor="k")
    ax.set_xlabel("Petal length (cm)")
    ax.set_ylabel("Petal width (cm)")
    ax.set_title(title)
    ax.legend()
    plt.show()

## Evaluación final del modelo elegido por validación cruzada

In [ ]:
best_name = tabla.loc[0, 'Modelo']
best_clf = clone(pipelines[best_name]).fit(X_train, y_train)
y_pred_best = best_clf.predict(X_test)
print('Modelo seleccionado por CV:', best_name)
print('Exactitud en test:', accuracy_score(y_test, y_pred_best))
print('F1 macro en test:', f1_score(y_test, y_pred_best, average='macro'))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_best, display_labels=iris.target_names)
plt.show()
plot_decision_regions(best_clf, X_train, y_train, f'Frontera: {best_name}')


## Extensión visual: regresión logística
Observe la frontera usando entrenamiento; no vuelva a seleccionar modelos mirando el test.

In [ ]:
comparison_model = clone(pipelines['LogisticRegression']).fit(X_train, y_train)
plot_decision_regions(comparison_model, X_train, y_train, 'Regresión logística: entrenamiento')


## Extensión visual: bosque aleatorio

In [ ]:
comparison_model = clone(pipelines['RandomForest']).fit(X_train, y_train)
plot_decision_regions(comparison_model, X_train, y_train, 'Bosque aleatorio: entrenamiento')


## Conclusiones y recomendaciones
- **No existe un clasificador universalmente mejor**: el rendimiento depende del problema y los datos.
- Usa **líneas base simples** (LogisticRegression, árboles) antes de modelos complejos.
- **Validación cruzada** y **métricas adecuadas** son clave para comparar de forma justa.